# 📊 퀀트 자동 분석 — Gemini 연동 v2

순서: 셀1 설치 → 셀2 설정 → 셀3 수집 → 셀4 차트 → 셀5 프롬프트 → 셀6 Gemini → 셀7 현황판

> ⚠️ 투자 자문 아님. 모든 손익 책임은 본인에게 있습니다.

In [ ]:
# 셀 1: 설치
!pip install pykrx pandas numpy google-generativeai matplotlib --quiet
!apt-get install -y fonts-nanum -qq
import matplotlib; matplotlib.font_manager._load_fontmanager(try_read_cache=False)
print('✅ 설치 완료')

In [ ]:
# 셀 2: 설정
# ── API 키 안전 로드 (코랩 비밀 저장소) ──
# 최초 1회만: 코랩 왼쪽 사이드바 🔑 아이콘 클릭
# → '새 보안 비밀 추가' → 이름: GEMINI_API_KEY → 값: 발급받은 키 붙여넣기
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
print('✅ API 키 로드 완료')

# 분석할 종목
TICKERS = [
    ("042700", "한미반도체"),
    # ("005930", "삼성전자"),
    # ("000660", "SK하이닉스"),
]
LOOKBACK_DAYS = 60
print(f'✅ 설정완료 | {[t[1] for t in TICKERS]}')


In [ ]:
# 셀 3: 데이터 수집 + 지표 계산
from pykrx import stock
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

def normalize_ohlcv(df):
    '""pykrx 버전별 컬럼 차이 자동 처리""'
    col = list(df.columns)
    # 한글 컬럼명 매핑 (버전 무관)
    rename = {}
    for c in col:
        if '시가' in c or 'Open' in c:  rename[c] = '시가'
        elif '고가' in c or 'High' in c: rename[c] = '고가'
        elif '저가' in c or 'Low'  in c: rename[c] = '저가'
        elif '종가' in c or 'Close' in c: rename[c] = '종가'
        elif '거래량' in c or 'Volume' in c: rename[c] = '거래량'
    df = df.rename(columns=rename)
    return df[['시가','고가','저가','종가','거래량']]

def get_stock_data(ticker, name, lookback=60):
    end   = datetime.today().strftime('%Y%m%d')
    start = (datetime.today() - timedelta(days=lookback*2)).strftime('%Y%m%d')
    try:
        raw = stock.get_market_ohlcv(start, end, ticker)
        df  = normalize_ohlcv(raw)
        df  = df[df['거래량'] > 0].tail(lookback).copy()
        if df.empty:
            print(f'❌ {name} 데이터 없음'); return None

        for n in [5, 20, 60, 120]:
            df[f'MA{n}'] = df['종가'].rolling(n).mean().round(0)

        df['BB_mid']   = df['종가'].rolling(20).mean()
        bb_std         = df['종가'].rolling(20).std()
        df['BB_upper'] = (df['BB_mid'] + 2*bb_std).round(0)
        df['BB_lower'] = (df['BB_mid'] - 2*bb_std).round(0)
        df['BB_mid']   = df['BB_mid'].round(0)

        delta = df['종가'].diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        df['RSI'] = (100 - 100/(1 + gain/loss.replace(0, np.nan))).round(1)

        ema12 = df['종가'].ewm(span=12, adjust=False).mean()
        ema26 = df['종가'].ewm(span=26, adjust=False).mean()
        df['MACD']      = (ema12 - ema26).round(0)
        df['Signal']    = df['MACD'].ewm(span=9, adjust=False).mean().round(0)
        df['MACD_hist'] = (df['MACD'] - df['Signal']).round(0)

        low10  = df['저가'].rolling(10).min()
        high10 = df['고가'].rolling(10).max()
        df['Sto_K'] = (100*(df['종가']-low10)/(high10-low10).replace(0,np.nan)).round(1)
        df['Sto_D'] = df['Sto_K'].rolling(5).mean().round(1)

        df['거래량_비율'] = (df['거래량']/df['거래량'].shift(1)*100).round(1)

        try:
            inv = stock.get_market_trading_value_by_date(start, end, ticker)
            inv.columns = [c.replace(' ','') for c in inv.columns]
            for c in inv.columns:
                if '외국인' in c: inv = inv.rename(columns={c:'외인순매수'})
                if '기관' in c and '합계' in c: inv = inv.rename(columns={c:'기관순매수'})
            keep = [c for c in ['외인순매수','기관순매수'] if c in inv.columns]
            df = df.join(inv[keep], how='left')
        except:
            df['외인순매수'] = np.nan
            df['기관순매수'] = np.nan

        print(f'✅ {name}({ticker}) 완료 | {len(df)}일치')
        return df
    except Exception as e:
        print(f'❌ {name} 오류: {e}'); return None

stock_data = {}
for ticker, name in TICKERS:
    stock_data[ticker] = {'name': name, 'df': get_stock_data(ticker, name, LOOKBACK_DAYS)}
print('\n🎯 수집 완료')

In [ ]:
# 셀 4: 차트 시각화
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

def plot_chart(df, name, ticker):
    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(4, 1, height_ratios=[3,1,1,1], hspace=0.1)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax4 = fig.add_subplot(gs[3], sharex=ax1)
    x = range(len(df))
    for i, (_, row) in enumerate(df.iterrows()):
        c = '#e74c3c' if row['종가'] >= row['시가'] else '#3498db'
        ax1.plot([i,i],[row['저가'],row['고가']], color=c, lw=1)
        ax1.bar(i, abs(row['종가']-row['시가']),
                bottom=min(row['종가'],row['시가']), color=c, width=0.7)
    for ma, c in [('MA5','#f39c12'),('MA20','#2ecc71'),('MA60','#9b59b6'),('MA120','#1abc9c')]:
        if ma in df.columns:
            ax1.plot(x, df[ma], color=c, lw=1.2, label=ma)
    ax1.plot(x, df['BB_upper'], 'gray', lw=0.8, ls='--', alpha=0.6)
    ax1.plot(x, df['BB_lower'], 'gray', lw=0.8, ls='--', alpha=0.6)
    ax1.fill_between(x, df['BB_upper'], df['BB_lower'], alpha=0.05, color='gray')
    cur = df['종가'].iloc[-1]
    ax1.set_title(f'{name} ({ticker})  현재가: {cur:,.0f}원', fontsize=14)
    ax1.legend(loc='upper left', fontsize=8)
    vc = ['#e74c3c' if df['종가'].iloc[i]>=df['시가'].iloc[i] else '#3498db' for i in range(len(df))]
    ax2.bar(x, df['거래량'], color=vc, alpha=0.7)
    ax2.set_ylabel('거래량')
    ax3.plot(x, df['MACD'],   color='#3498db', lw=1.2, label='MACD')
    ax3.plot(x, df['Signal'], color='#e74c3c', lw=1.2, label='Signal')
    hc = ['#e74c3c' if v>=0 else '#3498db' for v in df['MACD_hist']]
    ax3.bar(x, df['MACD_hist'], color=hc, alpha=0.5)
    ax3.axhline(0, color='gray', lw=0.5)
    ax3.legend(loc='upper left', fontsize=8)
    ax3.set_ylabel('MACD')
    ax4.plot(x, df['RSI'], color='#9b59b6', lw=1.2)
    ax4.axhline(70, color='#e74c3c', lw=0.8, ls='--', alpha=0.7)
    ax4.axhline(30, color='#3498db', lw=0.8, ls='--', alpha=0.7)
    ax4.set_ylim(0, 100)
    ax4.set_ylabel('RSI')
    step = max(1, len(df)//10)
    ax4.set_xticks(range(0, len(df), step))
    ax4.set_xticklabels([str(df.index[i])[:10] for i in range(0,len(df),step)], rotation=45, fontsize=8)
    for ax in [ax1,ax2,ax3]:
        plt.setp(ax.get_xticklabels(), visible=False)
    plt.tight_layout()
    plt.show()

for ticker, data in stock_data.items():
    if data['df'] is not None:
        plot_chart(data['df'], data['name'], ticker)
print('✅ 차트 완료')

In [ ]:
# 셀 5: Gemini 분석 텍스트 생성
def build_prompt(df, name, ticker):
    l = df.iloc[-1]; p = df.iloc[-2]
    w = df.iloc[-6] if len(df)>=6 else df.iloc[0]
    macd_sig = ('골든크로스' if l['MACD']>l['Signal'] and p['MACD']<=p['Signal'] else
                '데드크로스' if l['MACD']<l['Signal'] and p['MACD']>=p['Signal'] else
                'MACD>Signal(상승)' if l['MACD']>l['Signal'] else 'MACD<Signal(하락)')
    rsi_s = '과매수' if l['RSI']>=70 else '과매도' if l['RSI']<=30 else '중립'
    sto_s = '과매수' if l['Sto_K']>=80 else '과매도' if l['Sto_K']<=20 else '중립'
    bb_r  = l['BB_upper']-l['BB_lower']
    bb_p  = round((l['종가']-l['BB_lower'])/bb_r*100,1) if bb_r>0 else 50
    bb_s  = '상단근접' if bb_p>=80 else '하단근접' if bb_p<=20 else f'중간({bb_p}%)'
    f3 = df['외인순매수'].tail(3).sum() if '외인순매수' in df.columns else 'N/A'
    g3 = df['기관순매수'].tail(3).sum() if '기관순매수' in df.columns else 'N/A'
    cur  = l['종가']
    lines = [
        '=== 퀀트 자동 분석 데이터 ===',
        f'종목: {name} ({ticker}) | 분석일: {str(df.index[-1])[:10]}',
        '',
        f'현재가: {cur:,.0f}원 | 전일대비: {round((cur/p["종가"]-1)*100,2)}%',
        f'시가: {l["시가"]:,.0f} | 고가: {l["고가"]:,.0f} | 저가: {l["저가"]:,.0f}',
        '',
        f'MA5: {l["MA5"]:,.0f} | MA20: {l["MA20"]:,.0f} | MA60: {l["MA60"]:,.0f} | MA120: {l["MA120"]:,.0f}',
        f'현재가 MA5 {"위" if cur>l["MA5"] else "아래"} / MA20 {"위" if cur>l["MA20"] else "아래"}',
        '',
        f'BB 상단: {l["BB_upper"]:,.0f} | 중단: {l["BB_mid"]:,.0f} | 하단: {l["BB_lower"]:,.0f} | 위치: {bb_s}',
        '',
        f'MACD: {l["MACD"]:,.0f} / Signal: {l["Signal"]:,.0f} -> {macd_sig}',
        f'RSI(14): {l["RSI"]} -> {rsi_s} | Sto K: {l["Sto_K"]} D: {l["Sto_D"]} -> {sto_s}',
        '',
        f'거래량: {l["거래량"]:,}주 | 전일대비: {l["거래량_비율"]:.0f}% | 20일평균: {df["거래량"].tail(20).mean():,.0f}주',
        f'외인 3일순매수: {f"{f3:,.0f}원" if f3!="N/A" else "N/A"} | 기관: {f"{g3:,.0f}원" if g3!="N/A" else "N/A"}',
        '',
        '=== 분석 요청 (R:R 2.0이상 / 손절 -7% 적용) ===',
        '1. 추세 판정  2. 지지/저항  3. 매수 조건',
        '4. 손절가  5. 목표가(R:R포함)  6. 리스크  7. 최종판정',
    ]
    return '\n'.join(lines)

prompts = {}
for ticker, data in stock_data.items():
    if data['df'] is not None:
        prompts[ticker] = build_prompt(data['df'], data['name'], ticker)
        print(prompts[ticker])
        print('='*60)
print('✅ 프롬프트 완료')

In [ ]:
# 셀 6: Gemini API 호출 v4
import google.generativeai as genai
import os

os.environ['PYTHONIOENCODING'] = 'utf-8'

genai.configure(api_key=GEMINI_API_KEY)

# 사용 가능한 모델 목록 확인 (오류 시 참고용)
try:
    available = [m.name for m in genai.list_models()
                 if 'generateContent' in m.supported_generation_methods]
    flash_models = [m for m in available if 'flash' in m]
    print('사용가능 Flash 모델:', flash_models[:5])
    MODEL_NAME = 'models/gemini-2.5-flash'
except:
    MODEL_NAME = 'models/gemini-2.5-flash'
    print(f'모델명 사용: {MODEL_NAME}')

model = genai.GenerativeModel(MODEL_NAME)
print(f'모델 로드 완료: {MODEL_NAME}')

# 시스템 프롬프트는 영어로 — 한국어 답변은 본문에서 요청
SYSTEM = (
    'You are a Korean stock quantitative analysis AI. '
    'Always respond in Korean language. '
    'Apply these rules strictly: '
    'Reject trades with R:R below 2.0 / '
    'Stop-loss at -7% from entry / '
    'No new entries between 09:00-09:30 KST / '
    'Never average down on losing positions'
)

print('Gemini 분석 시작...')
for ticker, prompt in prompts.items():
    name = stock_data[ticker]['name']
    print(f'\n[{name}] 분석 결과')
    print('-'*60)
    try:
        full = SYSTEM + '\n\n' + prompt
        response = model.generate_content(full)
        # 코랩 환경 안전 출력
        print(response.text)
    except UnicodeEncodeError as e:
        # 유니코드 출력 오류 시 대체 출력
        safe = response.text.encode('utf-8', errors='replace').decode('utf-8')
        print(safe)
    except Exception as e:
        print(f'오류: {type(e).__name__}: {e}')
    print('='*60)


In [ ]:
# 셀 7: 관심종목 현황판 (컬럼 자동감지 버전)
from pykrx import stock as krx
from datetime import datetime, timedelta
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

WATCHLIST = [
    ("042700", "한미반도체"),
    ("005930", "삼성전자"),
    ("000660", "SK하이닉스"),
    ("012450", "한화에어로스페이스"),
    ("329180", "HD현대중공업"),
]

def safe_ohlcv(ticker, start, end):
    '"""컬럼 개수 무관하게 안전하게 OHLCV 추출"""'
    df = krx.get_market_ohlcv(start, end, ticker)
    rename = {}
    for c in df.columns:
        if '시가'  in c: rename[c] = '시가'
        elif '고가' in c: rename[c] = '고가'
        elif '저가' in c: rename[c] = '저가'
        elif '종가' in c: rename[c] = '종가'
        elif '거래량' in c: rename[c] = '거래량'
    df = df.rename(columns=rename)
    return df[['시가','고가','저가','종가','거래량']]

today = datetime.today().strftime('%Y%m%d')
start = (datetime.today() - timedelta(days=120)).strftime('%Y%m%d')
rows  = []

for ticker, name in WATCHLIST:
    try:
        df = safe_ohlcv(ticker, start, today)
        df = df[df['거래량']>0]
        if len(df)<20: continue
        l, p = df.iloc[-1], df.iloc[-2]
        ma5  = df['종가'].rolling(5).mean().iloc[-1]
        ma20 = df['종가'].rolling(20).mean().iloc[-1]
        delta= df['종가'].diff()
        gain = delta.clip(lower=0).rolling(14).mean()
        loss = (-delta.clip(upper=0)).rolling(14).mean()
        rsi  = (100 - 100/(1+gain/loss.replace(0,1e-10))).iloc[-1]
        chg  = (l['종가']/p['종가']-1)*100
        volr = l['거래량']/df['거래량'].tail(20).mean()*100
        sig = ''
        if rsi<=30:    sig += '📉과매도 '
        if rsi>=70:    sig += '📈과매수 '
        if volr>=200:  sig += '🔥거래량폭발 '
        if l['종가']>ma5>ma20: sig += '✅정배열 '
        if l['종가']<ma5<ma20: sig += '❌역배열 '
        if not sig:    sig = '➖중립'
        rows.append({
            '종목': name,
            '현재가': f'{l["종가"]:,.0f}',
            '등락(%)': f'{chg:+.2f}%',
            'RSI': f'{rsi:.1f}',
            'MA5': f'{ma5:,.0f}',
            'MA20': f'{ma20:,.0f}',
            '거래량비율': f'{volr:.0f}%',
            '신호': sig
        })
        print(f'✅ {name} 완료')
    except Exception as e:
        print(f'❌ {name} 오류: {e}')
        rows.append({'종목':name,'현재가':'ERR','등락(%)':str(e)[:30],
                     'RSI':'','MA5':'','MA20':'','거래량비율':'','신호':''})

print('\n📊 관심종목 현황판')
print('='*90)
print(pd.DataFrame(rows).to_string(index=False))
print('='*90)
print('신호 기준: 과매도 RSI≤30 / 과매수 RSI≥70 / 거래량폭발 200%이상')